## Agent Traces to Supervised Fine-tuning

This demo showcases how Agent Traces stored in App Insights can be used to perform Supervised Fine-tuning (SFT) on an AzureOpenAI model.

### Prerequisites

1. You'll need Owner or RBAC Administrator role on your Azure subscription to assign roles.
2. You'll need to Deploy the Hosted Agent to your Foundry project by following the README in this directory: [retail-agent-langgraph](./retail-agent-langgraph).

After deploying the agent in your Foundry project, confirm that there are enough agent traces to perform fine-tuning. In Foundry, navigate to `Agents > {your-agent} > Traces > Conversations`. There should be at least 10 conversations to proceed.


### Setup .env file

Copy the .env.template file to .env and update the placeholders with your Foundry project and App Insights details.

In [ ]:
!cp .env.template .env

### Install azure-ai-projects

In [4]:
%pip install azure_ai_projects-2.2.0-py3-none-any.whl

### Grant Foundry project access to App Insights

Foundry project service principal requires `Log Analytics Reader` role on the App Insights resource to read the agent traces. Grant the access by following these steps: 

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

FOUNDRY_PROJECT_ID = os.environ["FOUNDRY_PROJECT_ID"]
APPLICATION_INSIGHTS_ID = os.environ["APPLICATION_INSIGHTS_ID"]

Fetch Foundry Project System-Assigned Managed Identity's Principal ID

In [ ]:
!az resource show --ids {FOUNDRY_PROJECT_ID} --query identity.principalId -o tsv

Copy the above Principal ID in the below command:

In [ ]:
!az role assignment create --assignee "COPIED_PRINCIPAL_ID" --role "Log Analytics Reader" --scope {APPLICATION_INSIGHTS_ID}

### Start Data Generation Job

Start Data Generation Job to export Agent Traces in App Insights to a Supervised Fine-tuning file.

In [2]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential
)

In [ ]:
from datetime import datetime, timedelta, timezone

from azure.ai.projects.models import (
    DataGenerationJob,
    DataGenerationJobInputs,
    DataGenerationJobScenario,
    TracesDataGenerationJobOptions,
    TracesDataGenerationJobSource,
)

AGENT_NAME = "retail-agent-langgraph"
TRACES_START_TIME = datetime.now(timezone.utc) - timedelta(days=1)

options = TracesDataGenerationJobOptions(
    max_samples=50,  # maxinum number of samples in output files
    train_split=0.8  # split output into training and validation sets
)

source = TracesDataGenerationJobSource(
    agent_name=AGENT_NAME,
    start_time=int(TRACES_START_TIME.timestamp())
)

def _put_type_first(model):
    # TODO: Fix temporary SDK workaround for an API bug.
    if hasattr(model, "_data") and "type" in model._data:
        model._data = {"type": model._data["type"], **{k: v for k, v in model._data.items() if k != "type"}}

job = DataGenerationJob(
    inputs=DataGenerationJobInputs(
        name="agent_traces_sft_data",
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        options=_put_type_first(options),
        sources=[_put_type_first(source)]
    )
)

job = project_client.beta.datasets.create_generation_job(job)
print(f"Data generation job created: {job.id}")


In [ ]:
import time

from azure.ai.projects.models import JobStatus
from IPython.display import clear_output

while job.status not in [JobStatus.SUCCEEDED, JobStatus.FAILED, JobStatus.CANCELLED]:
    job = project_client.beta.datasets.get_generation_job(job.id)
    clear_output(wait=True)
    print(f"Job status: {job.status}")
    time.sleep(10)

print(f"Data generation job finished with status: {job.status}")
if job.status == JobStatus.FAILED:
    raise Exception("Data generation job failed with error:", job.error)

In [ ]:
from azure.ai.projects.models import DataGenerationJobOutputType

for output in job.result.outputs:
    assert output.type == DataGenerationJobOutputType.FILE

    print(f"Output {output.type}: File ID: {output.id}, Filename: {output.filename}")

Output file: File ID: file-68a0d0cd4a744cc6b1198279847bc5a8, Filename: None
Output file: File ID: file-ff70b44e3e5d485fa1ea2035a74ab974, Filename: None


In [20]:
file_ids = [output.id for output in job.result.outputs if output.type == DataGenerationJobOutputType.FILE]

training_file = file_ids[0]
validation_file = file_ids[1] if len(file_ids) > 1 else None

### Inspect Output Files

Inspect the output files from the Data Generation Job. They should be ready-to-use for fine-tuning an AzureOpenAI model.

In [3]:
training_file = "file-74c99f5a2f734d96b1aec343636c9e51"

In [ ]:
import json

openai_client = project_client.get_openai_client()

file_content = openai_client.files.content(training_file)
training_samples = [json.loads(line) for line in file_content.text.splitlines() if line.strip()]
training_sample = training_samples[0]
training_sample["messages"]


[{'role': 'user', 'content': 'Hi, I need help with a keyboard order.'},
 {'role': 'assistant',
  'content': 'Sure — I can help with that.\n\nPlease share your order ID and what you want to do with the keyboard:\n- return\n- exchange\n- replacement\n- cancellation\n- report as lost/late/damaged\n- defective issue\n\nIf it’s a return or exchange, please also tell me the reason, for example:\n- defective\n- changed mind\n- doesn’t fit your needs\n- wrong item\n- damaged in shipping\n- opened but not needed\n\nOnce I have the order ID and reason, I’ll check the order and eligibility.'},
 {'role': 'user', 'content': 'Hi, I need help with a keyboard order.'},
 {'role': 'assistant',
  'content': 'Sure — I can help with that.\n\nPlease share your order ID and what you want to do with the keyboard:\n- return\n- exchange\n- replacement\n- cancellation\n- report as lost/late/damaged\n- defective issue\n\nIf it’s a return or exchange, please also tell me the reason, for example:\n- defective\n- ch

In [15]:
tools = training_sample.get("tools", [])

for tool in tools:
    print(tool["function"]["name"], tool["function"]["parameters"])

get_order_details {'properties': {'order_id': {'type': 'string'}}, 'required': ['order_id'], 'type': 'object'}
get_fulfillment_status {'properties': {'order_id': {'type': 'string'}}, 'required': ['order_id'], 'type': 'object'}
check_resolution_policy {'properties': {'order_id': {'type': 'string'}, 'item_id': {'type': 'string'}, 'reason': {'type': 'string'}}, 'required': ['order_id', 'item_id', 'reason'], 'type': 'object'}
check_inventory {'properties': {'sku': {'type': 'string'}}, 'required': ['sku'], 'type': 'object'}
calculate_resolution {'properties': {'order_id': {'type': 'string'}, 'items': {'items': {}, 'type': 'array'}}, 'required': ['order_id', 'items'], 'type': 'object'}
submit_resolution {'properties': {'order_id': {'type': 'string'}, 'resolution_summary': {'type': 'string'}}, 'required': ['order_id', 'resolution_summary'], 'type': 'object'}


### Start Fine-tuning Job

In [ ]:
finetuning_job = openai_client.fine_tuning.jobs.create(
    model="gpt-4.1-mini-2025-04-14",
    training_file=training_file,
    validation_file=validation_file,
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
                "n_epochs": 3,
                "batch_size": "auto",
                "learning_rate_multiplier": "auto",
            },
        },
    },
    suffix="agent-traces-sft",
)

print(f"Fine-tuning job created: {finetuning_job.id}")


In [ ]:
while finetuning_job.status not in ["succeeded", "failed", "cancelled"]:
    events = client.fine_tuning.jobs.list_events(finetuning_job.id)
    clear_output(wait=True)
    print("Latest job events:")
    for event in events.data[-3:]:
        local_time = datetime.fromtimestamp(event.created_at).strftime("%Y-%m-%d %H:%M:%S")
        print("-", local_time, event.message)
    time.sleep(10)

    finetuning_job = client.fine_tuning.jobs.retrieve(finetuning_job.id)

print("Fine-tuning finished with status:", finetuning_job.status)
if finetuning_job.status == "failed":
    raise Exception("Fine-tuning job failed with error:", finetuning_job.error)